# Chạy Ollama trên Google Colab qua Cloudflare (KHÔNG CẦN TÀI KHOẢN)
Notebook này giúp bạn đưa phần nặng nhất của AI Chatbot lên GPU miễn phí của Google Colab mà không cần đăng ký.
**HƯỚNG DẪN:** Bạn chỉ cần bấm nút **PLAY (Chạy ô này)** ở ngay bên trái ô code dưới đây và đợi máy xử lý xong tất cả.

In [ ]:
import os
import time
import subprocess
import threading
import re

print("1. Đang cài đặt công cụ giải nén zstd...")
os.system("apt-get install -y zstd pciutils > /dev/null 2>&1")

print("2. Đang tải Ollama từ GitHub (Định dạng tar.zst mới nhất)...")
os.system("wget -q -O ollama-linux-amd64.tar.zst https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst")

print("3. Đang cài đặt Ollama vào hệ thống (Kích hoạt GPU)...")
os.system("tar --zstd -xf ollama-linux-amd64.tar.zst -C /usr")

ollama_path = "/usr/bin/ollama"
os.system(f"chmod +x {ollama_path}")

print("4. Đang khởi chạy máy chủ Ollama dưới nền...")
# TỐI ƯU HÓA: Bật Flash Attention, ép giữ model trong RAM vĩnh viễn, tải nhiều model cùng lúc
os.environ['OLLAMA_ORIGINS'] = '*'
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '3'

os.system(f"{ollama_path} serve > ollama.log 2>&1 &")
time.sleep(5)

health = subprocess.run(["curl", "-s", "http://127.0.0.1:11434/"], capture_output=True, text=True)
if "Ollama" not in health.stdout:
    time.sleep(10)

print("5. Đang tải đường hầm Cloudflare...")
os.system("wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
os.system("chmod +x cloudflared-linux-amd64")

def run_cloudflared():
    os.system("./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:11434 > cloudflare.log 2>&1")

threading.Thread(target=run_cloudflared, daemon=True).start()

print("6. Đang tạo link Public (Chờ 10 giây)...")
time.sleep(10)

url = None
try:
    with open('cloudflare.log', 'r') as f:
        content = f.read()
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
        if match:
            url = match.group(1)
except Exception as e:
    pass

print("\n==================================================")
if url:
    print(f"🔥 LINK KẾT NỐI (Hãy copy): {url}")
    print(f"\n👉 Dán vào file .env ở máy tính của bạn: OLLAMA_HOST={url}")
else:
    print("Đang tạo link, bạn hãy đợi vài giây rồi mở file 'cloudflare.log' ở cột bên trái của Colab để copy link nhé.")
print("==================================================\n")

print("7. Bắt đầu tải mô hình AI SIÊU TỐC (Mất khoảng 2-3 phút)...")
print("-> Đang kéo nomic-embed-text...")
os.system(f"{ollama_path} pull nomic-embed-text")

print("-> Đang kéo qwen2.5:3b (Phiên bản siêu nhẹ chạy cực nhanh)...")
os.system(f"{ollama_path} pull qwen2.5:3b")

print("\n✅ HOÀN TẤT! Google Colab đã sẵn sàng nhận tin nhắn từ máy của bạn.")
print("\n⚠️ KHÔNG tắt Tab này. Ô code sẽ chạy liên tục để giữ kết nối.")
while True:
    time.sleep(60)